In [8]:
!ls -lsah

total 204K
   0 drwxr-sr-x 5 solveit solveit  314 Nov 20 04:07 .
   0 drwxr-sr-x 3 solveit solveit   27 Nov 12 17:26 ..
   0 drwxr-sr-x 8 solveit solveit  257 Nov 19 03:19 .git
4.0K -rw-r--r-- 1 solveit solveit  478 Nov 18 23:08 .gitignore
   0 drwxr-xr-x 3 solveit solveit   70 Nov 19 03:17 .pytest_cache
 12K -rw-r--r-- 1 solveit solveit  12K Nov 15 00:44 LICENSE
4.0K -rw-r--r-- 1 solveit solveit 1.2K Nov 15 00:44 README.md
   0 drwxr-sr-x 2 solveit solveit   61 Nov 19 03:17 __pycache__
4.0K -rw-r--r-- 1 solveit solveit    9 Nov 19 03:19 batch_cmd.txt
148K -rw-r--r-- 1 solveit solveit 146K Nov 20 04:07 build_wish.ipynb
4.0K -rw-r--r-- 1 solveit solveit 1.7K Nov 19 03:17 build_wish.py
   0 -rw-r--r-- 1 solveit solveit    0 Nov 19 03:19 empty_batch_cmd.txt
4.0K -rw-r--r-- 1 solveit solveit   13 Nov 19 03:19 multiline_batch_cmd.txt
 20K -rwxr-xr-x 1 solveit solveit  17K Nov 20 04:03 wish
4.0K -rw-r--r-- 1 solveit solveit 1.3K Nov 20 04:06 wish.c


In [9]:
!git status


On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   build_wish.ipynb
	modified:   wish.c

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	__pycache__/
	build_wish.py

no changes added to commit (use "git add" and/or "git commit -a")


In [10]:
!git log --oneline

dca03e3 (HEAD -> main, origin/main) feature: made a composable fn for running shell in interactive and batch mode
514c9f1 doc: txt files for testing
b097881 doc: simple txt file to test batch mode
13af012 feature: added basic loop for shell
dddf2d0 docs: updated readme
0334887 Initial commit


## Problem 1 — A C program that prints its arguments

Create `wish.c` with a `main` function that receives `argc` and `argv`. Compile it into an executable named `wish`.

**Your task:** Print the argument count, then print each argument with its index, including `argv[0]`. This is only an argument-inspection program for now, not a shell.

**Completion checks:**
- Run `./wish` and identify what `argv[0]` contains.
- Run `./wish hello world` and check the count and order.
- Run `./wish "hello world"` and explain why it receives fewer arguments than the previous invocation.
- Compile with warnings enabled and resolve any warnings.

Keep building on this file in the following problems. Save a working version after each milestone.

In [11]:
%%writefile wish.c
#include <stdio.h>

int main(int argc, char *argv[]) {
    printf("argument count: %d\n", argc);

    for (int i = 0; i < argc; i++){
        printf("%d: %s\n", i, argv[i]);
    }


    return 0;
}

Overwriting wish.c


In [12]:
!gcc -Wall -Werror wish.c -o wish


In [13]:
!./wish hello world

argument count: 3
0: ./wish
1: hello
2: world


In [14]:
!./wish "hello world"

argument count: 2
0: ./wish
1: hello world


## Problem 2 — Read lines in an interactive loop

Replace the startup argument-printing behavior with a loop that prints the exact prompt `wish> ` and reads one line from standard input using `getline()`.

**Your task:** Temporarily echo each line back so you can verify input handling. Keep reading until EOF; then exit successfully. Make sure the prompt is visible before waiting for input.

**Completion checks:**
- Enter several lines and get a fresh prompt after each.
- Empty lines do not end the program.
- A very long line is read without being cut off by a fixed-size buffer.
- EOF exits with status 0.

No command execution yet. The echo is temporary debugging output.

In [15]:
%%writefile wish.c
#include <stdio.h>

int main(int argc, char *argv[]) {
    printf("argument count: %d\n", argc);

    for (int i = 0; i < argc; i++){
        printf("%d: %s\n", i, argv[i]);
    }
    while (1) {
        printf("wish> ");
        fflush(stdout);

    }


    return 0;
}

Overwriting wish.c


In [16]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[]) {

    char *line = NULL;
    size_t capacity = 0;

    while (1) {
        printf("wish> ");
        fflush(stdout);
        if (getline(&line, &capacity, stdin) == -1) break;
        printf("%s \n", line);
    }


    return 0;
}

Overwriting wish.c


In [17]:
!gcc -Wall -Werror wish.c -o wish

In [18]:
!printf 'hello\nworld\n' | ./wish

wish> hello
 
wish> world
 
wish> 

In [19]:
!printf '' | ./wish

wish> 

In [20]:
import subprocess

In [21]:
r = subprocess.run(
    ["./wish"],
    input="hello\n",
    capture_output=True,
    text=True,
    timeout=5)
r

CompletedProcess(args=['./wish'], returncode=0, stdout='wish> hello\n \nwish> ', stderr='')

In [22]:
r = subprocess.run(
    ["./wish"],
    input="hello\nhow\nare\nyou?",
    capture_output=True,
    text=True,
    timeout=5)
r

CompletedProcess(args=['./wish'], returncode=0, stdout='wish> hello\n \nwish> how\n \nwish> are\n \nwish> you? \nwish> ', stderr='')

In [23]:
r.stdout

'wish> hello\n \nwish> how\n \nwish> are\n \nwish> you? \nwish> '

In [24]:
print(r.stdout, end="")

wish> hello
 
wish> how
 
wish> are
 
wish> you? 
wish> 

In [25]:
def display_output(res: subprocess.CompletedProcess) -> None: print(res.stdout, end="")

In [26]:
display_output(r)

wish> hello
 
wish> how
 
wish> are
 
wish> you? 
wish> 

In [27]:
long_line = "x" * 100_000
r = subprocess.run(
    ["./wish"],
    input=long_line + "\n",
    capture_output=True,
    text=True,
    timeout=5,
    check=True
    )

In [28]:
display_output(r)

wish> ...
 
wish> 

In [29]:
r = subprocess.run(
    ["./wish"],
    input="hello\nhow\nare\nyou?",
    text=True,
    timeout=5)
r

CompletedProcess(args=['./wish'], returncode=0)

## Problem 3 — Select interactive or batch input

Use the startup arguments to choose an input source:
- No extra arguments: read standard input and print prompts.
- One extra argument: open that batch file and read it without printing prompts.
- More than one extra argument, or a batch file that cannot be opened: report an error and exit with status 1.

Use the same reading loop for both modes. EOF in either mode exits with status 0.

**Error rule from now on:** Every shell-detected error must write exactly `An error has occurred\n` to standard error, with `\n` meaning a newline. Do not add explanations or use a different error message. Check return values as you introduce operations that can fail.

**Completion checks:** Try a multi-line batch file, an empty file, a nonexistent file, and two filename arguments. Keep the temporary line echo only for debugging.

In [30]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>

int interactive_mode(){
    char *line = NULL;
    size_t capacity = 0;
    while (1){
        printf("wish> ");
        fflush(stdout);
        if (getline(&line, &capacity, stdin) == -1) break;
        printf("%s \n", line);
    }


    return 0;
}

int batch_mode(){

    printf("batch mode!!!!\n");
    return 0;
}


int main(int argc, char *argv[]) {

    if (argc == 1){ interactive_mode();}
    else { batch_mode();}


    return 0;
}

Overwriting wish.c


In [31]:
!gcc -Wall -Werror wish.c -o wish

In [32]:
!./wish hello

batch mode!!!!


In [33]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>

int write_error_message(void){
    char error_message[30] = "An error has occurred\n";
    write(STDERR_FILENO, error_message, strlen(error_message)); 

    return 0;
}

int interactive_mode(void){
    char *line = NULL;
    size_t capacity = 0;
    while (1){
        printf("wish> ");
        fflush(stdout);
        if (getline(&line, &capacity, stdin) == -1) break;
        printf("%s \n", line);
    }


    return 0;
}

int batch_mode(void){

    printf("batch mode!!!!\n");
    return 0;
}


int main(int argc, char *argv[]) {

    if (argc == 1){ interactive_mode();}
    else { batch_mode();}


    return 0;
}




Overwriting wish.c


In [34]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>

int write_error_message(void){
    char error_message[30] = "An error has occurred\n";
    write(STDERR_FILENO, error_message, strlen(error_message)); 

    return 0;
}

int interactive_mode(void){
    char *line = NULL;
    size_t capacity = 0;
    while (1){
        printf("wish> ");
        fflush(stdout);
        if (getline(&line, &capacity, stdin) == -1) break;
        printf("%s \n", line);
    }


    return 0;
}

int batch_mode(void){

    printf("batch mode!!!!\n");
    return 0;
}


int main(int argc, char *argv[]) {

    if (argc == 1){ interactive_mode();}
    else { batch_mode();}


    return 0;
}




Overwriting wish.c


In [35]:
!gcc -Wall -Werror wish.c -o wish

In [36]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>

int write_error_message(void){
    char error_message[30] = "An error has occurred\n";
    write(STDERR_FILENO, error_message, strlen(error_message)); 

    return 0;
}

int interactive_mode(void){
    char *line = NULL;
    size_t capacity = 0;
    while (1){
        printf("wish> ");
        fflush(stdout);
        if (getline(&line, &capacity, stdin) == -1) break;
        printf("%s \n", line);
    }


    return 0;
}

int batch_mode(void){

    printf("batch mode!!!!\n");
    return 0;
}


int main(int argc, char *argv[]) {

    if (argc == 1){ interactive_mode();}
    else if (argc == 2) { batch_mode();}
    else { write_error_message(); return 1;}


    return 0;
}




Overwriting wish.c


In [37]:
!gcc -Wall -Werror wish.c -o wish

In [38]:
%%writefile batch_cmd.txt
ls -lsah

Overwriting batch_cmd.txt


In [39]:
%%writefile multiline_batch_cmd.txt
ls -lsah
pwd

Overwriting multiline_batch_cmd.txt


In [40]:
from pathlib import Path
Path("empty_batch_cmd.txt").write_text("")

0

In [41]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>

int write_error_message(void){
    char error_message[30] = "An error has occurred\n";
    write(STDERR_FILENO, error_message, strlen(error_message)); 

    return 0;
}

int interactive_mode(void){
    char *line = NULL;
    size_t capacity = 0;
    while (1){
        printf("wish> ");
        fflush(stdout);
        if (getline(&line, &capacity, stdin) == -1) break;
        printf("%s \n", line);
    }


    return 0;
}

int batch_mode(const char *filename){

    printf("%s: batch mode!!!!\n", filename);
    return 0;
}


int main(int argc, char *argv[]) {

    if (argc == 1){ interactive_mode();}
    else if (argc == 2) { batch_mode(argv[1]);}
    else { write_error_message(); return 1;}


    return 0;
}




Overwriting wish.c


In [42]:
!gcc -Wall -Werror wish.c -o wish

In [43]:
!./wish batch_cmd.txt

batch_cmd.txt: batch mode!!!!


In [44]:
!./wish multiline_batch_cmd.txt

multiline_batch_cmd.txt: batch mode!!!!


In [45]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>

int write_error_message(void){
    char error_message[30] = "An error has occurred\n";
    write(STDERR_FILENO, error_message, strlen(error_message)); 

    return 0;
}

int interactive_mode(void){
    char *line = NULL;
    size_t capacity = 0;
    while (1){
        printf("wish> ");
        fflush(stdout);
        if (getline(&line, &capacity, stdin) == -1) break;
        printf("%s \n", line);
    }


    return 0;
}

int batch_mode(const char *filename){


    FILE *input = fopen(filename, "r");
    if (input == NULL) { return -1; }

    char *line = NULL;
    size_t capacity = 0;
    if (getline(&line, &capacity, input) == -1){ return -1;}
    printf("%s: batch mode!!!!\n", line);
    return 0;
}


int main(int argc, char *argv[]) {

    if (argc == 1){ interactive_mode();}
    else if (argc == 2) { batch_mode(argv[1]);}
    else { write_error_message(); return 1;}


    return 0;
}




Overwriting wish.c


In [46]:
!gcc -Wall -Werror wish.c -o wish

In [47]:
!./wish batch_cmd.txt

ls -lsah
: batch mode!!!!


In [48]:
!./wish multiline_batch_cmd.txt

ls -lsah
: batch mode!!!!


In [49]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>

int write_error_message(void){
    char error_message[30] = "An error has occurred\n";
    write(STDERR_FILENO, error_message, strlen(error_message)); 

    return 0;
}


static int run_shell(FILE *input, int display_prompt){
    char *line = NULL;
    size_t capacity = 0;

    while (1){
        if (display_prompt != 0){
            printf("wish>: ");
            fflush(stdout);
        }
        if (getline(&line, &capacity, input) == -1){ return -1;}
        printf("%s\n", line);

    }

    return 0;
}


int main(int argc, char *argv[]) {
    int display_prompt = 1;
    FILE *input = stdin;

    if (argc == 2) { 
        display_prompt = 0;
        input = fopen(argv[1], "r");

        if (input == NULL){ write_error_message(); return 1;}
    }
    else if (argc > 2){ write_error_message(); return 1;}

    run_shell(input, display_prompt);

    return 0;
}




Overwriting wish.c


In [50]:
!gcc -Wall -Werror wish.c -o wish

In [51]:
!./wish batch_cmd.txt

ls -lsah



In [52]:
!./wish multiline_batch_cmd.txt

ls -lsah

pwd



In [53]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>

int write_error_message(void){
    char error_message[30] = "An error has occurred\n";
    write(STDERR_FILENO, error_message, strlen(error_message)); 

    return 0;
}


static int run_shell(FILE *input, int display_prompt){
    char *line = NULL;
    size_t capacity = 0;

    while (1){
        if (display_prompt != 0){
            printf("wish> ");
            fflush(stdout);
        }
        if (getline(&line, &capacity, input) == -1){ break;}
        printf("%s", line);

    }

    free(line);
    if (ferror(input)){ return 1;}

    return 0;
}


int main(int argc, char *argv[]) {
    int display_prompt = 1;
    FILE *input = stdin;

    if (argc > 2) {
        write_error_message();
        return 1;
    }

    if (argc == 2) { 
        display_prompt = 0;
        input = fopen(argv[1], "r");

        if (input == NULL){ 
            write_error_message(); 
            return 1;
        }
    }

    int code = run_shell(input, display_prompt);

    if (code != 0){
        write_error_message();
        return 1; 
    }

    return 0;
}




Overwriting wish.c


In [54]:
!gcc -Wall -Werror wish.c -o wish

In [55]:
!./wish batch_cmd.txt

ls -lsah


In [56]:
!./wish multiline_batch_cmd.txt

ls -lsah
pwd


In [57]:
!./wish empty_batch_cmd.txt

In [58]:
!./wish empty_batch_cmd.txt multiline_batch_cmd.txt

An error has occurred


In [59]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>

int write_error_message(void){
    char error_message[30] = "An error has occurred\n";
    write(STDERR_FILENO, error_message, strlen(error_message)); 

    return 0;
}


static int run_shell(FILE *input, int display_prompt){
    char *line = NULL;
    size_t capacity = 0;

    while (1){
        if (display_prompt != 0){
            printf("wish> ");
            fflush(stdout);
        }
        if (getline(&line, &capacity, input) == -1){ break;}
        printf("%s", line);

    }

    free(line);
    if (ferror(input)){ return 1;}

    return 0;
}


int main(int argc, char *argv[]) {
    int display_prompt = 1;
    FILE *input = stdin;

    if (argc > 2) {
        write_error_message();
        return 1;
    }

    if (argc == 2) { 
        display_prompt = 0;
        input = fopen(argv[1], "r");

        if (input == NULL){ 
            write_error_message(); 
            return 1;
        }
    }

    int code = run_shell(input, display_prompt);
    if (argc == 2 && fclose(input) == EOF) code = 1;
    
    if (code != 0){
        write_error_message();
        return 1; 
    }

    return 0;
}




Overwriting wish.c


In [60]:
!gcc -Wall -Werror wish.c -o wish

In [61]:
!./wish batch_cmd.txt

ls -lsah


In [62]:
!./wish multiline_batch_cmd.txt

ls -lsah
pwd


In [63]:
!./wish empty_batch_cmd.txt

In [64]:
!./wish empty_batch_cmd.txt multiline_batch_cmd.txt

An error has occurred


In [65]:
!./wish non_existent.txt

An error has occurred


## Problem 4 — Turn a command line into arguments

Parse each input line into a command name and its arguments. Consider `strsep()` and how you will handle empty tokens.

**Your task:** Separate tokens on spaces, tabs, and the line-ending newline. Ignore whitespace-only input. Prepare an argument array with a terminating null pointer for later use with `execv()`.

Temporarily print the parsed tokens instead of echoing the entire line.

**Completion checks:**
- `ls -la /tmp` produces three tokens in order.
- Leading/trailing whitespace, repeated spaces, and tabs do not create unwanted empty arguments.
- A blank line causes no error.
- You have a clear plan for the lifetime of the input buffer and pointers into it.

Do not implement quoting, wildcard expansion, or variable expansion; the assignment does not require them. The quoted startup argument in Problem 1 was processed by the shell that launched your program.

In [72]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>

int write_error_message(void){
    char error_message[30] = "An error has occurred\n";
    write(STDERR_FILENO, error_message, strlen(error_message)); 

    return 0;
}


static int run_shell(FILE *input, int display_prompt){
    char *line = NULL;
    size_t capacity = 0;

    while (1){
        if (display_prompt != 0){
            printf("wish> ");
            fflush(stdout);
        }
        if (getline(&line, &capacity, input) == -1){ break;}
        char *cursor = line;
        while (cursor != NULL){
            char *token = strsep(&cursor, " \t\n");
            if (token[0] == '\0'){ continue;}
            printf("%s\n", token);
        }

    }

    free(line);
    if (ferror(input)){ return 1;}

    return 0;
}


int main(int argc, char *argv[]) {
    int display_prompt = 1;
    FILE *input = stdin;

    if (argc > 2) {
        write_error_message();
        return 1;
    }

    if (argc == 2) { 
        display_prompt = 0;
        input = fopen(argv[1], "r");

        if (input == NULL){ 
            write_error_message(); 
            return 1;
        }
    }

    int code = run_shell(input, display_prompt);
    if (argc == 2 && fclose(input) == EOF) code = 1;
    
    if (code != 0){
        write_error_message();
        return 1; 
    }

    return 0;
}




Overwriting wish.c


In [73]:
!gcc -Wall -Werror wish.c -o wish

In [74]:
!./wish batch_cmd.txt

ls
-lsah


## Problem 5 — Execute one external command and wait

Replace token-debugging output with execution. For this milestone only, look for commands under `/bin`.

**Your task:** Construct the executable pathname, check it with `access(..., X_OK)`, create a child with `fork()`, and execute the command in that child using `execv()`. The parent waits with `wait()` or `waitpid()` before reading another line.

Use `execv()`, not `system()` or a different exec-family function. If execution fails in the child, report the error and terminate that child; it must not become another shell loop.

**Completion checks:**
- `ls` and `ls -la /tmp` run correctly.
- `sleep 1` finishes before the next prompt appears.
- An unknown command reports the required error, and the shell accepts another command.
- A failed command does not terminate the parent shell.
- No temporary argument/token/line debugging output remains.

## Problem 6 — Implement `exit` and `cd`

Recognize built-ins before searching for an external executable. Run built-ins in the shell process itself.

**Your task:**
- `exit` with no arguments exits with status 0.
- `exit` with any arguments reports an error and leaves the shell running.
- `cd` requires exactly one argument and calls `chdir()`.
- Wrong argument counts or a failed `chdir()` report an error and leave the shell running.

**Completion checks:**
- Run `pwd`, then `cd /tmp`, then `pwd` again.
- Try `cd`, `cd /tmp /bin`, and `cd` with a nonexistent directory.
- Try `exit extra`, followed by a valid command, then `exit`.
- In a batch file, valid `exit` prevents subsequent lines from running.

Explain why running `cd` in a child would not accomplish the required behavior.

## Problem 7 — Maintain the shell's search path

Replace the hard-coded `/bin` lookup with a shell-owned list of directories. Initialize it to exactly one entry: `/bin`.

**Your task:** Implement the `path` built-in. It accepts zero or more directory arguments and replaces the entire previous list. Search directories in order for external commands using `access(..., X_OK)`.

An empty path disables external commands, but not built-ins. Keep path entries valid after the input buffer is reused or freed; release old entries when replacing the path.

**Completion checks:**
- `path /bin /usr/bin` allows commands to be found in either directory.
- `path` followed by `ls` reports an error; `cd`, `path`, and `exit` still work.
- `path /bin` restores external execution.
- Replacing the path does not retain old directories.
- `path /nonexistent /bin` still lets you find `ls` in the later directory.

Direct execution via absolute or relative command paths is not required.

## Problem 8 — Parse and validate output redirection

Extend parsing to recognize `>` even when there is no whitespace around it. Separate the command's argument list from an optional output filename.

**Your task:** Before launching a command, reject missing commands, missing output filenames, multiple `>` operators, and multiple tokens to the right of `>`.

**Completion checks:**
- Accept `ls > output`, `ls>output`, and `ls -la /tmp>output` as equivalent redirection forms.
- Reject `> output`, `ls >`, `ls > one two`, and `ls > one > two`.
- Blank input remains harmless.
- Invalid syntax reports the required error without launching the malformed command.

At this checkpoint, you are validating the parsed representation; actual file redirection comes next. Redirection for built-ins is outside the required tests.

## Problem 9 — Redirect both output streams

Use the parsed filename to redirect an external command's standard output **and standard error** to the same file.

**Your task:** In the child, open the destination for writing, creating it if needed and truncating it if it already exists. Connect the file to both output descriptors before `execv()`. Study `open()`, `dup2()`, and `close()` for this step, and handle failures.

Keep the parent shell's descriptors unchanged.

**Completion checks:**
- `echo hello>output` writes to the file, not the terminal.
- Repeating redirection with shorter output removes old trailing content.
- `ls /a-directory-that-does-not-exist > errors` puts the program's diagnostic in the file.
- A following command without redirection still prints normally.
- An output pathname in a nonexistent directory produces the required shell error, and the shell continues.

Do not replace diagnostics generated by an executed program with the shell's generic error message.

## Problem 10 — Parse a line into multiple commands

Add support for `&` as a separator between commands. It must work with or without surrounding whitespace.

**Your task:** Split the line into command segments, then apply your existing argument and redirection parsing to each segment independently. Keep parsing separate from launching so the next problem can change when waiting happens.

**Completion checks:**
- `echo one & echo two` yields two commands.
- `echo one&echo two` yields the same two commands.
- `echo one>first&echo two>second` associates each filename with the correct command.
- Spaces and tabs around separators do not become arguments.
- Empty or whitespace-only segments are handled safely; never dereference a missing command name.

The assignment does not specify every empty-segment edge case. Choose and document a consistent policy rather than inventing extra shell features. Do not implement pipes or interpret `&` as permission to skip waiting for the whole line.

## Problem 11 — Launch commands in parallel, then wait

Change execution so all external commands on one input line are launched before the shell waits for any of them.

**Your task:** Record the children you successfully create, then wait for all of them before printing the next prompt or reading the next batch line. Continue handling built-ins in the shell process, not as external programs.

**Completion checks:**
- `sleep 1 & sleep 1` takes roughly one second rather than two.
- `sleep 2 & echo ready` can print `ready` before the sleep finishes, but the shell waits for both before advancing.
- `echo one>first&echo two>second` produces the two separate files.
- An invalid command does not prevent already-started children from being reaped.
- A following batch line runs only after all children from the previous line finish.

Output order among parallel commands is not guaranteed. Do not add stronger ordering requirements than the assignment specifies.

## Problem 12 — Audit errors and resource lifetimes

Review the complete loop for error recovery, memory ownership, open files, and child processes. Fix gaps now rather than adding new features.

**Your task:**
- Check relevant operation results, including file opening, allocation, `fork()`, `execv()`, `chdir()`, redirection, and waiting.
- Distinguish EOF from a read failure.
- Use only the required message for shell-detected errors.
- Preserve startup failures' status 1 and normal EOF/valid `exit` status 0.
- Continue after ordinary command errors.
- Free per-line and replaced-path allocations when no longer needed.
- Close unneeded file descriptors and reap every successfully launched child.
- Ensure failed children cannot fall back into the shell loop.

**Completion checks:** Run many input lines, repeated `path` changes, long lines, invalid commands, and repeated redirections. Use compiler warnings and, if available, memory-checking tools to investigate leaks and invalid accesses. Keep previously working tests passing.

## Problem 13 — Run the final acceptance suite

Create repeatable tests covering the entire assignment. For each, record expected standard output, standard error, output-file contents, and exit status where relevant. Avoid depending on a particular ordering of parallel output.

**Required coverage:**
- Startup with zero, one, and too many arguments; unreadable/nonexistent and empty batch files.
- Exact interactive prompt `wish> `, no batch prompts, and graceful EOF in both modes.
- External commands with arguments; blank lines, tabs, repeated whitespace, long lines, and a final batch line without a newline.
- All built-ins, their argument rules, failed directory changes, and empty/replaced/ordered search paths.
- Valid and invalid redirection, truncation, both output streams, and unwritable destinations.
- Parallel execution, per-command redirection, and waiting before the next input line.
- Recovery after errors and no leftover debugging output.

**Project complete when:** All tests pass, the executable is named `wish`, execution uses `execv()` rather than `system()`, and the behavior matches the original specification. Keep the final source and tests under version control or save a clearly labeled working copy.

Pipes, job control, quoting, expansion, and direct absolute/relative executable paths are not required for this project.

In [8]:
! ls -lsah ../../ostep-homework

total 16K
4.0K drwxrwsr-x 33 solveit solveit 4.0K Feb  5  2026 .
4.0K drwxrwsr-x 45 solveit solveit 4.0K Nov 12 17:26 ..
   0 drwxrwsr-x  8 solveit solveit  211 Feb  5  2026 .git
8.0K -rw-rw-r--  1 solveit solveit 6.1K Feb  5  2026 README.md
   0 drwxrwsr-x  2 solveit solveit  131 Feb  5  2026 cpu-api
   0 drwxrwsr-x  2 solveit solveit   57 Feb  5  2026 cpu-intro
   0 drwxrwsr-x  2 solveit solveit   55 Feb  5  2026 cpu-sched
   0 drwxrwsr-x  2 solveit solveit   53 Feb  5  2026 cpu-sched-lottery
   0 drwxrwsr-x  2 solveit solveit   50 Feb  5  2026 cpu-sched-mlfq
   0 drwxrwsr-x  2 solveit solveit   51 Feb  5  2026 cpu-sched-multi
   0 drwxrwsr-x  2 solveit solveit   49 Feb  5  2026 dist-afs
   0 drwxrwsr-x  2 solveit solveit   58 Feb  5  2026 dist-nfs
   0 drwxrwsr-x  2 solveit solveit   36 Feb  5  2026 file-devices
   0 drwxrwsr-x  2 solveit solveit   77 Feb  5  2026 file-disks
   0 drwxrwsr-x  2 solveit solveit  210 Feb  5  2026 file-ffs
   0 drwxrwsr-x  2 solveit solveit   50 Feb  5 